# Ingesting new videos, step by step

This notebook takes newly downloaded camera files from `data/raw/` to fully
processed cases: registered in the manifest, then audio, transcript, prosody,
sync, camera views and 2D pose.

Each step says **what it does**, shows **the command-line call** that does the
same thing, then **does it** with the Python API (or runs that command for
you). Run the cells in order, top to bottom. Nothing is written to the manifest
until Step 6, and only if you set `DRY_RUN = False` in the setup cell.

**Kernel:** `md-utility` (it has pandas, openpyxl and ffprobe). Steps that
need the speech or pose environment run their command through `conda run`.

**Privacy:** the cases table holds learners' and patients' real names. This
notebook never displays them; please keep it that way if you add cells.

The command-line version of Steps 1-6 is one script:
`python scripts/ingest_raw.py` (dry run) and `python scripts/ingest_raw.py --write`.

## Setup

In [ ]:
import os, shutil, sqlite3, subprocess, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
from multidata import casematch, ingest, manifest
from multidata.paths import MANIFEST_PATH, RAW_DIR
import ingest_raw          # scripts/ingest_raw.py: the same steps as one command

DRY_RUN = True             # set False to actually register files in Step 6
CONDA = shutil.which("conda") or os.path.expanduser("~/miniconda3/bin/conda")

def db():
    return sqlite3.connect(f"file:{MANIFEST_PATH}?mode=ro", uri=True)

def run(cmd):
    """Show a command-line call, run it from the repo root, print the end of its output."""
    print("$", " ".join(cmd).replace(CONDA, "conda"))
    out = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    lines = (out.stdout + out.stderr).strip().splitlines()
    keep = [l for l in lines if "INFO" in l or "ERROR" in l or "WARN" in l or not l.startswith("20")]
    print("\n".join(keep[-12:]))
    if out.returncode:
        print(f"(exit code {out.returncode})")

print("repo:", ROOT, "| manifest:", MANIFEST_PATH, "| dry run:", DRY_RUN)

## Step 1. Put the files in `data/raw/`

Copy each downloaded camera file into `data/raw/` **exactly as named** by the
source system, `<date>-<time>-<recording_id>-<camera>.mp4`. No renaming, no
subfolders. A case normally has two files, one per camera.

Command line: `ls data/raw/`. This cell lists only the files the manifest
doesn't know yet.

In [ ]:
new_files = ingest_raw.unregistered()
for f in new_files:
    print(f.name, f"{f.stat().st_size / 1e6:.0f} MB")
print(len(new_files), "new file(s)")

## Step 2. Make sure the cases are in the manifest

A video can only be registered against a case the manifest already has. Cases
come from the source system's Excel export, one sheet per case. Importing only
adds cases it doesn't have; existing ones are never changed.

Command line: `python -m multidata.manifest path/to/export.xlsx`

In [ ]:
# Only if you have a new export:
# result = manifest.import_video_manifest_excel("path/to/export.xlsx")
# print(len(result["added"]), "cases added")

print(db().execute("SELECT COUNT(*) FROM cases").fetchone()[0], "cases in the manifest")

## Step 3. Check the file names

The name carries the recording date and time and the camera number, which is
how the file gets matched to its case. A badly named file stops here.

In [ ]:
for f in new_files:
    try:
        p = ingest.parse_filename(f)
        print(f.name, "->", p["date"], p["time"], "camera", p["camera"])
    except ValueError as e:
        print("BAD NAME:", e)

## Step 4. Match each file to its case

The match uses the recording time (to the minute) and the room the camera is
installed in (the `cameras` table). If it fails, the reason is usually a camera
missing from that table: add it with `manifest.set_camera_room(camera, room)`.

In [ ]:
matches = {}
for f in new_files:
    try:
        matches[f] = casematch.resolve(f, manifest_path=MANIFEST_PATH)
    except casematch.CaseMatchError as e:
        print("NO MATCH:", f.name, "-", e)
rows = [(f.name, case_id, *db().execute(
            "SELECT room_number, recording_start_time FROM cases WHERE case_id = ?",
            (case_id,)).fetchone()) for f, case_id in matches.items()]
pd.DataFrame(rows, columns=["file", "case", "room", "recording start"])

## Step 5. Inspect each file

`ffprobe` reads duration, frame rate, size and whether there's audio; a SHA-256
fingerprint catches a file that's already registered under another name.
Hashing takes a few seconds per file.

In [ ]:
info = {}
for f in matches:
    info[f] = ingest.summarize(f)
    dup = db().execute("SELECT case_id, camera FROM videos WHERE sha256 = ?",
                       (info[f]["sha256"],)).fetchone()
    info[f]["duplicate_of"] = f"{dup[0]}/{dup[1]}" if dup else ""
pd.DataFrame([{"file": f.name, "case": matches[f], "camera": i["camera"],
               "minutes": round(i["duration_s"] / 60, 1), "fps": round(i["fps"], 2),
               "size": f"{i['width']}x{i['height']}", "audio": i["has_audio"],
               "duplicate_of": i["duplicate_of"]} for f, i in info.items()])

## Step 6. Register them

Adds one row per file to the manifest's `videos` table, with every stage
`pending`. Skips duplicates. Set `DRY_RUN = False` in the setup cell and re-run
it first.

Command line: `python scripts/ingest_raw.py --write`

In [ ]:
new_cases = sorted({matches[f] for f in info if not info[f]["duplicate_of"]})
for f, i in info.items():
    if i["duplicate_of"]:
        print("skip (duplicate):", f.name)
        continue
    fields = {k: v for k, v in i.items() if k != "duplicate_of"}
    if DRY_RUN:
        print("would register", f.name, "->", matches[f])
    else:
        manifest.add_video(case_id=matches[f], path=MANIFEST_PATH, **fields)
        print("registered", f.name, "->", matches[f])
print("cases touched:", new_cases)

## Step 7. Check camera pairs

3D needs both cameras of a case. A case with one camera still gets transcripts,
prosody and 2D pose.

In [ ]:
q = ",".join("?" * len(new_cases))
pd.read_sql_query(f"SELECT case_id AS 'case', GROUP_CONCAT(camera) AS cameras, COUNT(*) AS n "
                  f"FROM videos WHERE case_id IN ({q}) GROUP BY case_id", db(), params=new_cases)

## Step 8. Run the processing stages

Each stage only works on videos still `pending` for it, so re-running is safe.
`--only <case>` limits a stage to one case. The order matters:

| Stage | Env | What it makes | Works on |
|---|---|---|---|
| `audio` | md-speech | 16 kHz WAV | one camera per case (the "audio camera"); the other is `skipped` |
| `asr` | md-speech | transcript with speakers (also runs diarization) | audio camera |
| `prosody` | md-speech | pitch, loudness, voice quality; Praat files | audio camera |
| `elan` | md-speech | ELAN `.eaf` for annotators, with a pitch tier | audio camera, after `prosody` |
| `sync` | md-speech | camera offsets + frame times | every camera |
| `views` | md-pose | when each camera moves, which view it holds | every camera, after `sync` |
| `pose` | md-pose (GPU) | 2D body keypoints | every camera; slow (~35 min per case) |

A case whose audio camera was already processed shows the new camera as
`skipped` for the first four stages; that's expected.

In [ ]:
for case in new_cases:
    for stage in ("audio", "asr", "prosody", "elan", "sync"):
        run([CONDA, "run", "-n", "md-speech", "python", "scripts/run_stage.py", stage, "--only", case])

In [ ]:
# views: one worker, oldest videos first (it adds to a shared view library)
run([CONDA, "run", "-n", "md-pose", "python", "scripts/run_stage.py", "views"])

Pose takes the GPU for a long time. This cell checks for a pose job that's
already running and won't start a second one; when the GPU is free it starts
pose **in the background** (it keeps going if you close the notebook) and tells
you where to watch it. For a big batch, see `docs/operations.md`.

In [ ]:
busy = subprocess.run(["pgrep", "-f", "run_stage.py pose"], capture_output=True, text=True).stdout.split()
if busy:
    print("A pose job is already running (PIDs", ", ".join(busy), "). Run this cell again when it's done.")
else:
    cmd = [CONDA, "run", "-n", "md-pose", "python", "scripts/run_stage.py", "pose", "--accel-device", "cuda"]
    print("$", " ".join(cmd).replace(CONDA, "conda"))
    subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    print("started in the background; watch it with:  tail -f logs/run_stage.log")

## Step 9. Check the results

Every stage's status for the new cases (`done`, `skipped`, `failed`,
`pending`), then what each run cost (the `stage_runs` table feeds
`docs/cost_estimate.md`). A `failed` row's reason is in `logs/run_stage.log`.

In [ ]:
cols = ["audio", "asr", "prosody", "elan", "sync", "views", "pose"]
pd.read_sql_query(
    f"SELECT case_id AS 'case', camera, {', '.join(f'{c}_status AS {c}' for c in cols)} "
    f"FROM videos WHERE case_id IN ({q}) ORDER BY case_id, camera", db(), params=new_cases)

In [ ]:
pd.read_sql_query(
    f"SELECT stage, case_id AS 'case', camera, status, ROUND(wall_s) AS seconds, "
    f"ROUND(realtime_x, 1) AS realtime_x FROM stage_runs WHERE case_id IN ({q}) ORDER BY id",
    db(), params=new_cases)

## If something goes wrong

- **NO MATCH, camera has no known room:** add it, `manifest.set_camera_room("<camera>", "<room>")`, then re-run from Step 4.
- **NO MATCH, no case at that time:** the case isn't in the manifest yet (Step 2), or the recording started in a different minute than the export says.
- **A stage `failed`:** read the last lines of `logs/run_stage.log`. Fix the cause, set that row back to `pending` (`manifest.update(case, camera, **{"<stage>_status": "pending"})`) and re-run the stage.
- **"can't import its module here":** the stage was run in the wrong env; the table in Step 8 says which.